In [44]:
!pip install -q langchain langchain-community langchain-groq langchain-text-splitters faiss-cpu pypdf sentence-transformers

In [45]:
import os
os.environ["GROQ_API_KEY"] = "**************************"

In [46]:
from google.colab import files
uploaded = files.upload()

Saving HARSHADA KESTE.pdf to HARSHADA KESTE.pdf


In [51]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

# 1️⃣ Load PDF
loader = PyPDFLoader("HARSHADA KESTE.pdf")   # change name if different
documents = loader.load()

# 2️⃣ Split text into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = splitter.split_documents(documents)

# 3️⃣ Create embeddings (FREE)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 4️⃣ Store in FAISS vector DB
vectorstore = FAISS.from_documents(docs, embeddings)

# 5️⃣ Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

# 6️⃣ Load Groq LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [52]:
def ask_resume(question):

    # 🔎 Retrieve relevant chunks (NEW METHOD)
    docs = retriever.invoke(question)

    # Combine context
    context = "\n\n".join([d.page_content for d in docs])

    # Prompt to LLM
    prompt = f"""
    Answer the question using the resume context below.
    If answer not found, say 'Not in resume'.

    Resume:
    {context}

    Question: {question}
    """

    # 🤖 LLM response
    response = llm.invoke(prompt)
    print("\n🤖 Answer:\n", response.content)

In [53]:
ask_resume("What are the candidate skills?")


🤖 Answer:
 Based on the resume, the candidate's skills are:

1. Programming Languages: C++, Python, Java
2. Web Technologies: HTML, CSS, JavaScript, FastAPI, React
3. Core Concepts: Data Structures & Algorithms, OOP, DBMS
4. Databases: MySQL, Oracle
5. Tools: Git, GitHub, VS Code, Selenium
6. Soft Skills: Problem Solving, Team Collaboration, Analytical Thinking, Adaptability
